# Fine-tuning Classifier — Qwen2.5-7B + Unsloth + QLoRA

Ce notebook fine-tune **Qwen2.5-7B-Instruct** sur `classifier_dataset.json`
(produit par `scripts/prepare_dataset.py`) pour étiqueter un commentaire
Instagram avec son **T-type** (T1, T2, T2b, T3a, T3b, T4 ou T5).

**Différences clés vs le notebook generator** :
- L'output est un **token unique** (le label T-type) → `MAX_SEQ_LENGTH=256` suffit.
- La validation finale impose une sortie **strictement** ∈ `{T1, T2, T2b, T3a, T3b, T4, T5}` —
  un texte libre = échec d'apprentissage.
- Le `Modelfile` Ollama force un `stop "\n"` immédiat après le label pour
  empêcher le modèle de continuer à générer du contexte parasite.

**Cible matérielle** : RunPod / Google Colab Pro avec **GPU A100 40 GB**
(ou A6000 24 GB). Avec `MAX_SEQ_LENGTH=256` au lieu de 512, le training
est **~2× plus rapide** et la VRAM consommée tombe à ~14 Go — un T4 16 Go
suffit même.

**Pipeline complet** :
1. `python scripts/prepare_dataset.py` (locale)
2. Upload `scripts/data/classifier_dataset.json` à côté de ce notebook
3. Exécuter ce notebook → modèle pushé sur HuggingFace
4. Conversion GGUF → import Ollama sur le Mac Mini (cf. dernière cellule)

## 1. Installation

`unsloth` embarque déjà une version compatible de `transformers`, `peft`,
`bitsandbytes`, `xformers` et `triton` — on installe ensuite `trl` et
`accelerate` au cas où l'environnement Colab/RunPod ne les fournit pas.

> Sur **Google Colab**, redémarrer le runtime après cette cellule
> (Runtime → Restart) si une dépendance refuse de se charger.

In [ ]:
!pip install unsloth
!pip install trl transformers accelerate bitsandbytes

import torch
assert torch.cuda.is_available(), "GPU CUDA introuvable — bascule sur un runtime A100/T4."
print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
print(f"VRAM dispo : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

## 2. Configuration

Hyper-paramètres adaptés à un objectif **classification** (output court,
distribution sur un set fini de 7 labels).

- `MAX_SEQ_LENGTH=256` — l'output est **un seul token** (T-type), seul
  l'input (commentaire + niche + métriques) consomme du contexte. 256 tokens
  couvrent largement les commentaires Instagram (≤ 100 tokens en pratique).
- `LORA_R=16` / `LORA_ALPHA=32` — identique au generator, expressivité
  raisonnable pour une tâche de classification où les sous-distinctions
  (T2 vs T2b, T3a vs T3b) demandent quand même de la finesse.
- Effective batch size = **16**.
- 3 epochs : suffisant pour un dataset court (>200 paires) sans overfit.

> Adapter `HF_REPO` à ton compte HuggingFace avant exécution. `HF_TOKEN`
> doit être disponible (variable d'env ou `notebook_login()`).

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "./qwen_classifier_finetuned"
DATASET_PATH = "./classifier_dataset.json"
HF_REPO = "ton-username/aitertainment-classifier"

MAX_SEQ_LENGTH = 256  # output court (1 T-type) + input commentaire ≤ 200 tokens
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 2e-4

# Set fini de labels valides — utilisé par la cellule de test pour valider
# que le modèle répond bien un T-type et pas du texte libre.
VALID_TTYPES = {"T1", "T2", "T2b", "T3a", "T3b", "T4", "T5"}

print(f"Modèle de base : {MODEL_NAME}")
print(f"Dataset       : {DATASET_PATH}")
print(f"Sortie        : {OUTPUT_DIR}")
print(f"Repo HF       : {HF_REPO}")
print(
    f"Effective batch size : {BATCH_SIZE * GRAD_ACCUMULATION} "
    f"(BATCH_SIZE={BATCH_SIZE} × GRAD_ACCUMULATION={GRAD_ACCUMULATION})"
)
print(f"Labels valides : {sorted(VALID_TTYPES)}")

## 3. Chargement du modèle Qwen2.5-7B + injection des adapters LoRA

Unsloth charge le modèle en **4-bit** (NF4 quantization) puis injecte des
adapters LoRA sur les 7 modules d'attention/MLP de chaque bloc transformer.
Seuls les ~40 M paramètres LoRA sont entraînés (vs 7 B en plein), ce qui
permet le fine-tuning sur 14-40 Go de VRAM.

`use_gradient_checkpointing=True` recalcule les activations forward pour
réduire la VRAM (×2 sur la mémoire activations, coût : ~20 % de temps).

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing=True,
)

model.print_trainable_parameters()

## 4. Chargement du dataset et formatage Alpaca

Chaque entrée du dataset classifier est :

```json
{"instruction": "Classifie ce commentaire Instagram selon les types T1→T5.",
 "input": "Commentaire: ...\nNiche: ...\nVues: ...\nRatio comments/likes: ...",
 "output": "T2"}
```

L'`output` est un **token unique** (le label). On affiche aussi la
distribution des labels — un dataset très déséquilibré (ex : 90 % T2)
produira un modèle qui prédit toujours T2 avec confiance.

> **Le template Alpaca utilisé ici doit être réutilisé tel quel à
> l'inférence** (cf. cellule 7) — sinon le modèle ne reconnaît pas le prompt.

In [ ]:
import json
from collections import Counter

from datasets import Dataset

with open(DATASET_PATH) as f:
    data = json.load(f)

print(f"Paires chargées : {len(data)}")
assert len(data) >= 50, (
    f"Dataset trop petit ({len(data)} paires) — collecte plus de données via "
    "le bot Telegram (validations) avant fine-tuning."
)

# Distribution des labels — sanity check anti-déséquilibre.
label_counts = Counter(p["output"] for p in data)
print("\nDistribution des T-types :")
for label in sorted(VALID_TTYPES):
    pct = 100 * label_counts.get(label, 0) / len(data)
    print(f"  {label:<4} : {label_counts.get(label, 0):>4} ({pct:>5.1f} %)")

# Warning si une classe domine > 70 % — modèle dégénéré probable.
top_label, top_count = label_counts.most_common(1)[0]
if top_count / len(data) > 0.70:
    print(
        f"\n⚠️  Distribution très déséquilibrée (label {top_label} = "
        f"{100 * top_count / len(data):.0f} %) — envisager un sous-échantillonnage "
        "ou collecter plus de profils dans les autres T-types."
    )


def format_alpaca(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }


dataset = Dataset.from_list(data)
dataset = dataset.map(format_alpaca)

print("\nExemple formaté :")
print(dataset[0]["text"])
print(f"\nNombre de tokens (approx) sur l'exemple 0 : "
      f"{len(tokenizer.encode(dataset[0]['text']))}")

## 5. Entraînement (SFTTrainer)

`fp16=True` est correct sur A100/T4/V100 ; sur Ampere/Hopper récents
(A100, H100), `bf16=True` est plus stable — décommenter au besoin.

**Durée indicative** sur A100 40 GB (avec `MAX_SEQ_LENGTH=256`) :
- 200 paires × 3 epochs ≈ **1 min**
- 1 000 paires × 3 epochs ≈ **4 min**
- 5 000 paires × 3 epochs ≈ **18 min**

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        fp16=True,
        # bf16=True,  # à privilégier sur A100/H100
        logging_steps=10,
        save_steps=100,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
    ),
)

trainer_stats = trainer.train()
print("\nEntraînement terminé.")
print(f"Loss finale : {trainer_stats.training_loss:.4f}")

## 6. Sauvegarde locale + push HuggingFace

`save_pretrained` sauvegarde uniquement les **adapters LoRA** (~150 Mo)
plus le tokenizer — pas les 14 Go du modèle de base. C'est ce qu'on push
sur HF, ce qui économise bande passante et stockage.

> Côté inférence (Ollama / llama.cpp), il faudra **merger les adapters**
> dans le base model pour produire un `.gguf` autonome (cf. cellule
> "Intégration Mac Mini").

**Authentification HuggingFace** : un `HF_TOKEN` avec scope `write` doit
être disponible — sur Colab, exécuter `from huggingface_hub import notebook_login; notebook_login()`
en amont. Sur RunPod, exporter `HF_TOKEN` en variable d'env.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapters LoRA + tokenizer sauvegardés : {OUTPUT_DIR}")

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True, private=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"Modèle pushé sur HuggingFace : https://huggingface.co/{HF_REPO}")

## 7. Test rapide — validation stricte du format de sortie

Sanity check **anti-régression** : le modèle fine-tuné doit produire un
**T-type valide** et **uniquement** ça. Si la sortie contient du texte libre
(« Ce commentaire est un T2 parce que… »), le fine-tuning n'a pas convergé
et le `Modelfile` Ollama produira de la sortie inutilisable.

On exécute 5 cas de test (un par registre principal), puis on parse la
sortie via `_extract_ttype` qui n'accepte qu'un label strict ∈ `VALID_TTYPES`.
Le test final affiche un récap `accuracy = N/5` — < 4/5 = collecter plus
de données ou augmenter `EPOCHS`.

`temperature=0.0` (greedy decoding) : on veut un classifier **déterministe**,
pas créatif. `max_new_tokens=8` suffit pour le label le plus long (« T2b »
= 3 tokens).

In [ ]:
import re

from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


def _build_prompt(commentaire: str, niche: str, vues: int | str, ratio: float | str) -> str:
    """Construit le prompt Alpaca exact du dataset classifier."""
    return f"""### Instruction:
Classifie ce commentaire Instagram selon les types T1→T5.

### Input:
Commentaire: {commentaire}
Niche: {niche}
Vues: {vues}
Ratio comments/likes: {ratio}

### Response:
"""


def _extract_ttype(generated: str) -> str | None:
    """Parse la sortie du modèle ; retourne le T-type ou ``None`` si invalide.

    Le modèle peut générer ``"T2"`` (parfait) ou ``"T2\\n..."`` (premier mot
    correct, puis bavardage = ``stop`` mal configuré côté Ollama). On
    capture donc le **premier match** d'un pattern T-type ; tout le reste
    est considéré comme bruit. Si aucun pattern ne matche, le fine-tuning
    n'a pas convergé sur le format strict — alerte.
    """
    # Pattern : T1, T2, T2b, T3a, T3b, T4, T5 (T2b/T3a/T3b avant T2/T3 pour
    # éviter le greedy-match qui s'arrêterait à "T2" sur "T2b").
    m = re.search(r"\b(T2b|T3a|T3b|T1|T2|T3|T4|T5)\b", generated)
    if not m:
        return None
    candidate = m.group(1)
    return candidate if candidate in VALID_TTYPES else None


def classify(commentaire: str, niche: str, vues, ratio) -> tuple[str | None, str]:
    """Retourne ``(t_type, raw_output)`` — ``t_type=None`` si format invalide."""
    prompt = _build_prompt(commentaire, niche, vues, ratio)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,           # greedy = déterministe
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return _extract_ttype(raw), raw


# 5 cas de test couvrant les registres principaux. Les `expected` sont des
# attentes raisonnables — un dataset de qualité doit reproduire ces labels.
test_cases = [
    {
        "commentaire": "mdr trop vrai j'en peux plus",
        "niche": "humour",
        "vues": 250000,
        "ratio": 0.08,
        "expected": "T2",
    },
    {
        "commentaire": "bravo le débutant on sent l'expérience",
        "niche": "cuisine",
        "vues": 180000,
        "ratio": 0.12,
        "expected": "T3b",
    },
    {
        "commentaire": "merci pour cette belle leçon de vie",
        "niche": "développement personnel",
        "vues": 80000,
        "ratio": 0.04,
        "expected": "T1",
    },
    {
        "commentaire": "tg le clown personne te calcule",
        "niche": "gaming",
        "vues": 90000,
        "ratio": 0.20,
        "expected": "T3a",
    },
    {
        "commentaire": "viens chez Renault si t'aimes les pannes",
        "niche": "f1",
        "vues": 320000,
        "ratio": 0.15,
        "expected": "T4",
    },
]

print("=" * 70)
print("Test classifier (greedy, max_new_tokens=8)")
print("=" * 70)

correct = 0
invalid_format = 0
for i, tc in enumerate(test_cases, 1):
    t_type, raw = classify(tc["commentaire"], tc["niche"], tc["vues"], tc["ratio"])
    status = "✓" if t_type == tc["expected"] else (
        "✗ INVALID FORMAT" if t_type is None else "✗"
    )
    if t_type == tc["expected"]:
        correct += 1
    if t_type is None:
        invalid_format += 1
    print(f"\n[{i}] {tc['commentaire']!r}")
    print(f"    expected={tc['expected']:<4} | got={t_type or '(invalid)':<8} | raw={raw!r}")
    print(f"    {status}")

print("\n" + "=" * 70)
print(f"Accuracy : {correct}/{len(test_cases)}")
print(f"Outputs hors format strict : {invalid_format}/{len(test_cases)}")
print("=" * 70)

assert invalid_format == 0, (
    f"{invalid_format}/{len(test_cases)} outputs hors format strict — "
    "le modèle ne respecte pas la contrainte 'output = un seul T-type'. "
    "Augmente EPOCHS, ou collecte plus de données par classe."
)

## 8. Intégration Mac Mini (Ollama)

Pour utiliser le classifier fine-tuné dans le pipeline AItertainment, il
faut le **convertir au format GGUF** puis l'**importer dans Ollama** sous
le nom `aitertainment-classifier`. Cinq étapes, toutes sur le Mac Mini M4
(sauf 1 et 2 qui peuvent rester sur la machine A100 pour éviter de
télécharger 14 Go).

### Étape 1 — Merger les adapters LoRA dans le modèle de base

```python
# À exécuter en fin de notebook (avant de fermer la session GPU)
# ou dans un script séparé après pull du repo HF.
from unsloth import FastLanguageModel

model.save_pretrained_merged(
    "./qwen_classifier_merged",
    tokenizer,
    save_method="merged_16bit",  # FP16 — meilleure qualité avant quantization GGUF
)
```

### Étape 2 — Conversion GGUF avec `llama.cpp`

```bash
# Sur le Mac Mini (ou la machine GPU avant download)
brew install cmake
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
make -j

python convert_hf_to_gguf.py ../qwen_classifier_merged \
  --outfile ../aitertainment-classifier-f16.gguf \
  --outtype f16

./quantize ../aitertainment-classifier-f16.gguf \
  ../aitertainment-classifier-q4km.gguf Q4_K_M
```

Sur Apple Silicon, `Q4_K_M` (~4.5 Go) est le sweet spot. Pour un classifier
court, on peut même viser `Q3_K_M` (~3.5 Go) sans perte notable — la
classification est moins sensible à la quantization que la génération
créative.

### Étape 3 — Créer un `Modelfile` Ollama

> Le `SYSTEM` ici est **différent** du generator : on liste explicitement
> les T-types attendus et on **interdit** toute sortie autre que le label.
> Le `TEMPLATE` Alpaca strict + le `PARAMETER stop "\n"` garantissent que
> le modèle s'arrête **immédiatement** après le label, sans générer de
> contexte parasite (ce qui casserait le parsing côté Python).

Créer `~/aitertainment/Modelfile.classifier` :

```
FROM ./aitertainment-classifier-q4km.gguf

SYSTEM """Tu es un classifieur de commentaires Instagram.
Tu classes chaque commentaire selon ces 7 types :
- T1  : Admiration sincère, encouragements, relation parasociale
- T2  : Humour tribal, communauté niche aimée, phrases récurrentes
- T2b : Humour participatif, les gens continuent le sketch dans les commentaires
- T3a : Haine directe non dissimulée
- T3b : Second degré invisible, moquerie que le créateur ne perçoit pas
- T4  : Identité communautaire forte, rituel de niche
- T5  : Créateur provocateur conscient qui exploite la haine

Tu réponds UNIQUEMENT par un label parmi : T1, T2, T2b, T3a, T3b, T4, T5.
Aucun autre mot. Pas de phrase. Pas d'explication. Pas de ponctuation finale."""

# Format Alpaca strict — DOIT matcher le template du notebook (cellules 4 et 7).
TEMPLATE """### Instruction:
{{ .System }}

### Input:
{{ .Prompt }}

### Response:
"""

# Greedy decoding : un classifier doit être déterministe.
PARAMETER temperature 0
PARAMETER top_p 1
PARAMETER repeat_penalty 1.0

# Output court : « T2b » = 3 tokens max ; on coupe agressivement.
PARAMETER num_predict 8

# Stops critiques — sans ça le modèle continue à générer du contexte
# parasite après le label, ce qui casse le parser Python.
PARAMETER stop "\n"
PARAMETER stop "###"
PARAMETER stop " "
PARAMETER stop "."
PARAMETER stop ","
```

> **Pourquoi `stop " "`?** Tous les T-types sont des tokens **uniques sans
> espace** (« T2b » est un seul token chez Qwen). Si le modèle commence à
> générer un espace, c'est qu'il dérive vers une phrase complète — on
> stoppe immédiatement.

### Étape 4 — Importer dans Ollama

```bash
ollama create aitertainment-classifier -f ~/aitertainment/Modelfile.classifier
ollama list   # vérifier que le modèle apparaît

# Smoke test direct CLI
ollama run aitertainment-classifier <<EOF
Commentaire: mdr trop vrai
Niche: humour
Vues: 250000
Ratio comments/likes: 0.08
EOF
# → doit répondre "T2" (et rien d'autre)
```

### Étape 5 — Mettre à jour `.env` du projet

Le projet utilise **un seul** `OLLAMA_MODEL` actuellement, partagé entre
le classifier et le generator. Pour le **mode bicéphale** (un modèle par
tâche), ajouter une variable séparée et adapter `modules/classifier.py` :

```bash
# Dans c:\Users\paulm\Documents\ASUS\AItertainment\aitertainment\.env
# (Mac Mini : ~/aitertainment/.env)

# Modèle par défaut (classifier — utilisé par CommentClassifier)
OLLAMA_MODEL=aitertainment-classifier

# Modèle generator (utilisé par generate_comments). Si non défini,
# fallback sur OLLAMA_MODEL pour rétro-compat.
OLLAMA_GENERATOR_MODEL=aitertainment-generator
```

Côté code, dans `modules/classifier.py`, faire pointer `generate_comments`
sur `OLLAMA_GENERATOR_MODEL` quand la variable existe :

```python
# Pseudo-diff dans generate_comments :
body["model"] = getattr(config, "OLLAMA_GENERATOR_MODEL", config.OLLAMA_MODEL)
```

### Vérification finale

```bash
cd ~/aitertainment
source .venv/bin/activate

# 1. Le SYSTEM et le TEMPLATE sont bons ?
ollama show aitertainment-classifier --modelfile

# 2. Le classifier renvoie bien un T-type strict via le pipeline Python ?
python -c "
from modules.classifier import CommentClassifier
out = CommentClassifier().classify(
    ['mdr trop vrai', 'le passage 0:08', 'on est foutu'],
    niche='humour'
)
print(out)
assert out['type'] in {'T1','T2','T2b','T3a','T3b','T4','T5'}, out
print('OK : classifier renvoie un T-type strict.')
"
```

Si la sortie n'est pas un T-type valide, augmenter `EPOCHS` à 5 et
collecter plus de paires par classe via le bot Telegram (validations
humaines + collecte automatique post-validation par `dataset_builder.py`).